# GraphRAG: City Semantic Search with Vector Similarity

This notebook demonstrates **semantic search** and **GraphRAG** on the Travel Recommendations graph:

1. **Generate embeddings** from `cityDescription` (text) and store them in City nodes
2. **Vector similarity search** to find cities similar to a query or reference city
3. **GraphRAG** — combine vector search with graph traversal for richer recommendations (activities, attractions, nearby cities)
4. **Natural language** — use an LLM to answer questions using retrieved graph context

**Prerequisites:**
- Neo4j Aura instance with the Travel Recommendations graph loaded
- OpenAI API key for embeddings

## Step 0: Setup Environment

In [ ]:
# Install dependencies (run once)
# pip install -r requirements.txt

In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

load_dotenv(override=True)

uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
user = os.getenv("NEO4J_USER", "neo4j")
database = os.getenv("NEO4J_DATABASE", "neo4j")
password = os.getenv("NEO4J_PASSWORD", "password")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

embedding_model = OpenAIEmbeddings()

with GraphDatabase.driver(uri, auth=(user, password)) as driver:
    driver.verify_connectivity()
    print("✓ Connected to Neo4j")

✓ Connected to Neo4j


## Step 1: Create Vector Index from City Descriptions

Use `Neo4jVector.from_existing_graph` to:
- Read `cityDescription` (and optionally `name`, `country`) from City nodes
- Generate embeddings via OpenAI
- Store embeddings in the `cityEmbedding` property
- Create a vector index for similarity search

In [2]:
vectorstore = Neo4jVector.from_existing_graph(
    embedding=embedding_model,
    url=uri,
    username=user,
    password=password,
    database=database,
    index_name="city_semantic_index",
    node_label="City",
    embedding_node_property="cityEmbedding",
    text_node_properties=["name", "country", "cityDescription"]
)

print("✓ Vector index 'city_semantic_index' created/updated from City nodes")

✓ Vector index 'city_semantic_index' created/updated from City nodes


## Step 2: Simple Similarity Search

Find cities semantically similar to a text query (e.g. "romantic city with art and museums").

In [3]:
query = "romantic city with art, museums and beautiful architecture"
k = 5

results = vectorstore.similarity_search_with_score(query, k=k)

print(f"Cities similar to: '{query}'\n")
for doc, score in results:
    print(f"[score: {score:.4f}] {doc.page_content[:150]}..." if len(doc.page_content) > 150 else f"    {doc.page_content}")
    print()

Cities similar to: 'romantic city with art, museums and beautiful architecture'

[score: 0.9222] 
name: Prague
country: Czech Republic
cityDescription: Prague is a fairy-tale city with stunning medieval architecture, beautiful Charles Bridge, hist...

[score: 0.9219] 
name: Paris
country: France
cityDescription: Paris is the romantic capital of France, known for its iconic Eiffel Tower, world-class museums like the...

[score: 0.9192] 
name: Venice
country: Italy
cityDescription: Venice is a magical floating city built on canals, famous for its romantic gondola rides, stunning St. M...

[score: 0.9186] 
name: Rome
country: Italy
cityDescription: Rome is the eternal city, home to ancient history with the Colosseum and Roman Forum, stunning Vatican Cit...

[score: 0.9184] 
name: Verona
country: Italy
cityDescription: Verona is the city of Romeo and Juliet, known for its Roman arena, beautiful old town, and romantic atmo...



## Step 3: Find Cities Similar to a Reference City

Use the description of a known city (e.g. Paris) as the query to find other cities with a similar profile.

In [4]:
reference_city = "Paris"
k = 5

# Get Paris description from the graph to use as query
with GraphDatabase.driver(uri, auth=(user, password)) as driver:
    records, _, _ = driver.execute_query(
        """
        MATCH (c:City {name: $cityName})
        RETURN c.name + ': ' + c.cityDescription AS queryText
        """,
        cityName=reference_city,
        database_=database
    )
    query_text = records[0]["queryText"] if records else reference_city

results = vectorstore.similarity_search_with_score(query_text, k=k + 1)

# Exclude the reference city itself (use helper: metadata lacks name, it's in page_content)
similar = [(doc, score) for doc, score in results if reference_city not in doc.page_content][:k]

print(f"Cities similar to {reference_city}:\n")
for doc, score in similar:
    print(f"[score: {score:.4f}] {doc.page_content[:150]}..." if len(doc.page_content) > 150 else f"    {doc.page_content}")
    print()

Cities similar to Paris:

[score: 0.9261] 
name: Lyon
country: France
cityDescription: Lyon is France's gastronomic capital, famous for its excellent cuisine, historic old town, beautiful arch...

[score: 0.9255] 
name: Bordeaux
country: France
cityDescription: Bordeaux is France's wine capital, known for its beautiful architecture, world-class vineyards, excel...

[score: 0.9244] 
name: Madrid
country: Spain
cityDescription: Madrid is Spain's vibrant capital, known for its royal palaces, world-renowned art museums like the Prad...

[score: 0.9242] 
name: Prague
country: Czech Republic
cityDescription: Prague is a fairy-tale city with stunning medieval architecture, beautiful Charles Bridge, hist...

[score: 0.9223] 
name: Nantes
country: France
cityDescription: Nantes is a vibrant French city on the Loire, known for its creative culture, excellent museums, beauti...



## Step 4: GraphRAG — Vector Search + Graph Traversal

Combine **vector similarity** with **graph traversal** to enrich results:
- Find cities similar to a query
- Traverse the graph to get: activities, attractions, nearby cities, events

This is the core of **GraphRAG**: retrieval augmented by graph context.

In [5]:
# Custom retrieval query: enrich each city with graph context (activities, attractions, nearby cities)
# The vector index provides (node, score); we traverse from node to get graph context
retrieval_query = """
    OPTIONAL MATCH (node)-[:OFFERS_ACTIVITY]->(a:Activity)
    OPTIONAL MATCH (node)-[:HAS_ATTRACTION]->(attr:Attraction)
    OPTIONAL MATCH (node)-[:NEARBY_TO]->(nearby:City)
    WITH node, score,
         collect(DISTINCT a.name) AS activities,
         collect(DISTINCT attr.name) AS attractions,
         collect(DISTINCT nearby.name) AS nearbyCities
    RETURN node.name + ', ' + node.country + ': ' + node.cityDescription AS text,
           score,
           node {name: node.name, country: node.country, averageBudget: node.averageBudget,
                 activities: activities[0..5], attractions: attractions[0..5], nearbyCities: nearbyCities[0..5]} AS metadata
"""

contextualized_vectorstore = Neo4jVector.from_existing_index(
    embedding_model,
    url=uri,
    username=user,
    password=password,
    database=database,
    index_name="city_semantic_index",
    retrieval_query=retrieval_query,
)

print("✓ Contextualized vector store created (GraphRAG retrieval)")

✓ Contextualized vector store created (GraphRAG retrieval)


In [6]:
query = "beach destination with good nightlife and culture"
k = 3

graphrag_results = contextualized_vectorstore.similarity_search_with_score(query, k=k)

print(f"GraphRAG results for: '{query}'\n")
for doc, score in graphrag_results:
    meta = doc.metadata
    print(f"📍 {meta.get('name')} ({meta.get('country')}) — score: {score:.4f}")
    print(f"   Budget: {meta.get('averageBudget', 'N/A')}")
    print(f"   Activities: {meta.get('activities', [])}")
    print(f"   Attractions: {meta.get('attractions', [])}")
    print(f"   Nearby: {meta.get('nearbyCities', [])}")
    print()

GraphRAG results for: 'beach destination with good nightlife and culture'

📍 Mykonos (Greece) — score: 0.9096
   Budget: high
   Activities: ['Beach', 'Nightlife']
   Attractions: ['Mykonos Windmills', 'Paradise Beach']
   Nearby: []

📍 San Sebastian (Spain) — score: 0.9017
   Budget: medium
   Activities: ['Beach', 'Relaxation']
   Attractions: ['La Concha Beach']
   Nearby: []

📍 Barcelona (Spain) — score: 0.8989
   Budget: medium
   Activities: ['Museums', 'Architecture', 'Monuments']
   Attractions: ['Sagrada Familia', 'Park Guell', 'Casa Batllo', 'Picasso Museum', 'Magic Fountain']
   Nearby: ['Madrid', 'Seville', 'Valencia', 'Granada', 'Cordoba']



## Step 5: Natural Language Q&A with GraphRAG

Use an LLM to answer natural language questions using the retrieved graph context.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def graphrag_response(question: str, k: int = 3) -> str:
    """Retrieve relevant cities + graph context, then generate a response."""
    docs = contextualized_vectorstore.similarity_search_with_score(question, k=k)
    context = "\n\n".join([
        f"City: {d.metadata.get('name')} ({d.metadata.get('country')})\n"
        f"Description: {d.page_content[:200]}...\n"
        f"Budget: {d.metadata.get('averageBudget')}\n"
        f"Activities: {d.metadata.get('activities', [])}\n"
        f"Attractions: {d.metadata.get('attractions', [])}\n"
        f"Nearby cities: {d.metadata.get('nearbyCities', [])}"
        for d, _ in docs
    ])
    return context

template = """You are a travel recommendation assistant. The user asked:
"{question}"

Based on the following cities retrieved from the graph (with their activities, attractions, and nearby cities), provide a helpful, concise recommendation. Mention specific cities and why they match.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel advisor."),
    ("human", template)
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
chain = prompt | llm | StrOutputParser()

In [8]:
question = "I want a city like Barcelona — vibrant, with beaches, good food and culture, but with a lower budget"

context = graphrag_response(question, k=4)
response = chain.invoke({"question": question, "context": context})

print("Question:", question)
print("\n---\n")
print("Response:", response)

Question: I want a city like Barcelona — vibrant, with beaches, good food and culture, but with a lower budget

---

Response: Based on your desire for a vibrant city with beaches, good food, and culture, but at a lower budget than Barcelona, I recommend considering **Lisbon, Portugal** and **San Sebastian, Spain**.

1. **Lisbon, Portugal**:
   - **Vibrancy**: Lisbon is a lively city known for its colorful neighborhoods and friendly atmosphere.
   - **Beaches**: While not directly on the beach, it's a short train ride to beautiful coastal areas like Cascais and Estoril.
   - **Food**: The city offers delicious cuisine, including the famous pastéis de nata, and has a variety of affordable dining options.
   - **Culture**: Lisbon is rich in history and culture, with attractions like Belém Tower and the Alfama District, where you can explore museums and historic sites.
   - **Budget**: Lisbon is generally more affordable than Barcelona, making it a great choice for budget-conscious travel

## Step 6: Create Similarity Relationships (Optional)

Store `SIMILAR_TO` relationships between cities based on vector similarity, so you can traverse them in Cypher without re-running embeddings.

In [40]:
from neo4j import GraphDatabase

def _get_city_name_country(doc):
    """Extract name and country from doc (metadata or page_content)."""
    name = doc.metadata.get("name")
    country = doc.metadata.get("country")
    if name is None or country is None:
        for line in doc.page_content.split("\n"):
            if line.startswith("name: "):
                name = line[6:].strip()
            elif line.startswith("country: "):
                country = line[9:].strip()
    return name or "N/A", country or ""

def create_similarity_relationships(threshold: float = 0.85, k_per_city: int = 5):
    """Create SIMILAR_TO relationships between cities based on embedding similarity."""
    with GraphDatabase.driver(uri, auth=(user, password)) as driver:
        records, _, _ = driver.execute_query(
            """
            MATCH (c:City)
            WHERE c.cityEmbedding IS NOT NULL
                                    RETURN c.cityId AS cityId, c.name AS name
            """,
            database_=database
        )

    created = 0
    for record in records:
        city_name = record["name"]
        similar = vectorstore.similarity_search_with_score(city_name, k=k_per_city + 1)
        # Exclude self
        similar = [(d, s) for d, s in similar if _get_city_name_country(d)[0] != city_name and s >= threshold][:k_per_city]

        for doc, score in similar:
            other_name = _get_city_name_country(doc)[0]
            with GraphDatabase.driver(uri, auth=(user, password)) as driver:
                driver.execute_query(
                    """
                    MATCH (a:City {name: $name1}), (b:City {name: $name2})
                    MERGE (a)-[r:SIMILAR_TO]->(b)
                    SET r.similarity = $score
                    """,
                    {"name1": city_name, "name2": other_name, "score": float(score)},
                    database_=database
                )
            created += 1

    return created

# Uncomment to run (creates many relationships)
created = create_similarity_relationships(threshold=0.80, k_per_city=3)
print(f"Created {created} SIMILAR_TO relationships")

Created 202 SIMILAR_TO relationships


## Summary

| Step | What it does |
|------|--------------|
| 1 | Create vector index from `cityDescription` (embeddings stored in `cityEmbedding`) |
| 2 | Simple similarity search by text query |
| 3 | Find cities similar to a reference city (e.g. Paris) |
| 4 | **GraphRAG**: Vector search + graph traversal (activities, attractions, nearby cities) |
| 5 | Natural language Q&A using retrieved graph context |
| 6 | Optional: Create `SIMILAR_TO` relationships for Cypher traversal |